# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a reproducible workflow for loading and exploring the FAIR² dataset on second primary colorectal cancer using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets and their fields by @id
record_sets_ids = [record_set['@id'] for record_set in getattr(metadata, 'recordSet', [])]
if not record_sets_ids:
    # Try to infer record sets from dataset
    record_sets_ids = dataset.record_sets()

print('Record sets and their fields (referenced by @id):')
record_set_fields = {}
for record_set_id in record_sets_ids:
    # Get record set object
    record_set = dataset.record_set(record_set_id)
    field_ids = [field['@id'] for field in getattr(record_set, 'field', [])]
    print(f"- Record Set @id: {record_set_id}")
    print(f"  Fields: {field_ids}")
    record_set_fields[record_set_id] = field_ids
    print()
if not record_sets_ids:
    print('No record sets found in metadata.')

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from available record sets into dataframes
# Use the record_set_ids and field ids as obtained above.
import warnings
dataframes = {}

if record_sets_ids:
    for record_set_id in record_sets_ids:
        try:
            records = list(dataset.records(record_set=record_set_id))
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded record set: {record_set_id}, shape: {dataframes[record_set_id].shape}")
        except Exception as ex:
            warnings.warn(f"Could not load records for {record_set_id}: {ex}")

    # Display the available columns in each dataframe
    for record_set_id, df in dataframes.items():
        print(f"\nColumns in DataFrame for record set {record_set_id}:")
        print(list(df.columns))
        display(df.head())
else:
    print('No record sets available to extract data from.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, categorizing data, removing outliers, and grouping data by important attributes for further analysis.

In [ ]:
# Demonstrate EDA on a main record set. Adjust @ids as per dataset structure from overview step.

# Let's try to use the first available record set and choose a numeric field for demo
from pandas.api.types import is_numeric_dtype
if dataframes:
    record_set_id = list(dataframes.keys())[0]  # Use the first one
    df = dataframes[record_set_id]
    # Try to find a numeric field
    numeric_cols = [col for col in df.columns if is_numeric_dtype(df[col])]
    if not numeric_cols:
        print('No numeric fields detected in first record set. Showing all columns for manual selection:')
        print(list(df.columns))
        numeric_field_id = None
    else:
        numeric_field_id = numeric_cols[0]

    # If a numeric field is found, perform standardization and filtering
    if numeric_field_id:
        threshold = 10 if df[numeric_field_id].max() > 20 else df[numeric_field_id].median()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Z-score normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to find a categorical/groupable field
        categorical_cols = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
        if categorical_cols:
            group_field_id = categorical_cols[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
        else:
            print('No suitable categorical grouping field found.')
    else:
        print('No numeric fields detected; please select a field manually based on displayed columns.')
else:
    print('No DataFrame loaded to perform EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Basic visualization on the selected (main) DataFrame
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    df = dataframes[record_set_id]
    if numeric_field_id and numeric_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field_id].dropna(), kde=True)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.show()
    if 'group_field_id' in locals() and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print('No DataFrame loaded for visualization.')

## 6. Conclusion
Summarized above are the loading, overview, basic processing, and visualization steps using [mlcroissant](https://pypi.org/project/mlcroissant/) with this FAIR² colorectal cancer dataset.

- Using Croissant metadata and the `@id` reference approach, you can programmatically explore and process rich tabular and structured datasets.
- Typical EDA operations can be performed with the dataframes loaded from each record set.
- This template supports extension to domain-specific analyses, modeling, or further visualization depending on fields present in each record set.
